In [ ]:
!pip install -q tensorflow-model-optimization

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 5.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# TMLP_QAT.py
# Temporal MLP with Quantization-Aware Training
# ============================================================

import os
import glob
import time

import numpy as np
import tensorflow as tf
import tensorflow_model_optimization as tfmot


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASET_PATH = "/content/drive/MyDrive/T-MLP/processed_data"

TRAIN_PATH = os.path.join(DATASET_PATH, "train")
VAL_PATH = os.path.join(DATASET_PATH, "validation")
TEST_PATH = os.path.join(DATASET_PATH, "test")

EXPECTED_TRAIN = 800
EXPECTED_VAL = 100
EXPECTED_TEST = 100

WINDOW_SIZE = 256
HOP_SIZE = 128

BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.01

MODEL_SAVE_PATH = "/content/TMLP_QAT.keras"


# ============================================================
# 2. ENVIRONMENT
# ============================================================

print("\nEnvironment")
print("----------------------------------------")
print("TensorFlow:", tf.__version__)
print("TFMOT     :", tfmot.__version__)


# ============================================================
# 3. DATASET VERIFICATION
# ============================================================

print("\nDataset verification")
print("----------------------------------------")

train_files = sorted(
    glob.glob(os.path.join(TRAIN_PATH, "*.npz"))
)

val_files = sorted(
    glob.glob(os.path.join(VAL_PATH, "*.npz"))
)

test_files = sorted(
    glob.glob(os.path.join(TEST_PATH, "*.npz"))
)

print(f"Training samples   : {len(train_files)}")
print(f"Validation samples : {len(val_files)}")
print(f"Test samples       : {len(test_files)}")

if len(train_files) != EXPECTED_TRAIN:
    raise ValueError(
        f"Expected {EXPECTED_TRAIN} training samples, "
        f"found {len(train_files)}"
    )

if len(val_files) != EXPECTED_VAL:
    raise ValueError(
        f"Expected {EXPECTED_VAL} validation samples, "
        f"found {len(val_files)}"
    )

if len(test_files) != EXPECTED_TEST:
    raise ValueError(
        f"Expected {EXPECTED_TEST} test samples, "
        f"found {len(test_files)}"
    )

print("Dataset verification successful.")


# ============================================================
# 4. LOAD SLIDING WINDOWS
# ============================================================

def load_windows(file_list):

    X = []
    Y = []

    for file_path in file_list:

        data = np.load(file_path)

        noisy = data["noisy_speech"].astype(np.float32)
        clean = data["clean_speech"].astype(np.float32)

        if len(noisy) != len(clean):
            raise ValueError(
                f"Length mismatch in {file_path}"
            )

        for start in range(
            0,
            len(noisy) - WINDOW_SIZE + 1,
            HOP_SIZE
        ):

            end = start + WINDOW_SIZE

            X.append(noisy[start:end])
            Y.append(clean[start:end])

    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)

    return X, Y


# ============================================================
# 5. LOAD DATA
# ============================================================

print("\nLoading training data...")

X_train, Y_train = load_windows(train_files)

print(
    f"Training windows: {len(X_train)}"
)


print("\nLoading validation data...")

X_val, Y_val = load_windows(val_files)

print(
    f"Validation windows: {len(X_val)}"
)


print("\nLoading test data...")

X_test, Y_test = load_windows(test_files)

print(
    f"Test windows: {len(X_test)}"
)


# ============================================================
# 6. TF.DATA PIPELINE
# ============================================================

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, Y_train)
)

train_dataset = (
    train_dataset
    .shuffle(
        buffer_size=len(X_train),
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, Y_val)
)

val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


# ============================================================
# 7. BUILD QAT-COMPATIBLE T-MLP
# ============================================================
#
# IMPORTANT:
#
# We intentionally DO NOT use a custom Keras Layer here.
#
# Instead:
#
# Dense -> Dense -> Add
#
# is explicitly constructed using standard Keras layers.
#
# This allows TFMOT to recognize and quantize the layers.
#
# ============================================================

def build_tmlp_qat_base():

    inputs = tf.keras.Input(
        shape=(WINDOW_SIZE,),
        name="noisy_audio"
    )

    # --------------------------------------------------------
    # Input:
    # (batch, 256)
    #
    # Reshape:
    # (batch, 256, 1)
    # --------------------------------------------------------

    x = tf.keras.layers.Reshape(
        (WINDOW_SIZE, 1),
        name="input_reshape"
    )(inputs)

    # --------------------------------------------------------
    # Feature projection
    #
    # 1 -> 64
    # --------------------------------------------------------

    x = tf.keras.layers.Dense(
        64,
        activation="relu",
        name="feature_projection"
    )(x)

    # ========================================================
    # TEMPORAL MLP BLOCK 1
    # ========================================================

    residual_1 = x

    x = tf.keras.layers.Dense(
        64,
        activation="relu",
        name="block1_dense1"
    )(x)

    x = tf.keras.layers.Dense(
        64,
        activation=None,
        name="block1_dense2"
    )(x)

    x = tf.keras.layers.Add(
        name="block1_residual_add"
    )([
        residual_1,
        x
    ])

    # ========================================================
    # TEMPORAL MLP BLOCK 2
    # ========================================================

    residual_2 = x

    x = tf.keras.layers.Dense(
        64,
        activation="relu",
        name="block2_dense1"
    )(x)

    x = tf.keras.layers.Dense(
        64,
        activation=None,
        name="block2_dense2"
    )(x)

    x = tf.keras.layers.Add(
        name="block2_residual_add"
    )([
        residual_2,
        x
    ])

    # ========================================================
    # OUTPUT PROJECTION
    # ========================================================

    x = tf.keras.layers.Dense(
        32,
        activation="relu",
        name="output_projection"
    )(x)

    x = tf.keras.layers.Dense(
        1,
        activation=None,
        name="audio_output"
    )(x)

    # --------------------------------------------------------
    # (batch, 256, 1)
    #
    # ->
    #
    # (batch, 256)
    # --------------------------------------------------------

    outputs = tf.keras.layers.Reshape(
        (WINDOW_SIZE,),
        name="output_reshape"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="TMLP_QAT"
    )

    return model


# ============================================================
# 8. BUILD BASE MODEL
# ============================================================

print("\nBuilding T-MLP QAT model...")
print("----------------------------------------")

base_model = build_tmlp_qat_base()

print("\nBase model:")
base_model.summary()


# ============================================================
# 9. APPLY QUANTIZATION-AWARE TRAINING
# ============================================================

print("\nApplying Quantization-Aware Training...")
print("----------------------------------------")

quantize_model = (
    tfmot.quantization.keras.quantize_model
)

qat_model = quantize_model(
    base_model
)

print("\nQAT model:")
qat_model.summary()


# ============================================================
# 10. COMPILE
# ============================================================

optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

qat_model.compile(
    optimizer=optimizer,
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)


# ============================================================
# 11. TRAIN
# ============================================================

print("\nStarting QAT training")
print("----------------------------------------")

start_training = time.perf_counter()

history = qat_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    verbose=1
)

training_time = (
    time.perf_counter()
    - start_training
)

print("\nTraining completed.")

print(
    f"Training time: "
    f"{training_time:.2f} seconds"
)


# ============================================================
# 12. TEST EVALUATION
# ============================================================

print("\nEvaluating on TEST dataset")
print("----------------------------------------")

test_loss, test_mae = qat_model.evaluate(
    X_test,
    Y_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

test_rmse = np.sqrt(test_loss)

print(
    f"\nTest MSE  : "
    f"{test_loss:.8f}"
)

print(
    f"Test RMSE : "
    f"{test_rmse:.8f}"
)

print(
    f"Test MAE  : "
    f"{test_mae:.8f}"
)


# ============================================================
# 13. GENERATE TEST PREDICTIONS
# ============================================================

print("\nGenerating test predictions...")
print("----------------------------------------")

start_prediction = time.perf_counter()

Y_pred = qat_model.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

prediction_time = (
    time.perf_counter()
    - start_prediction
)

print(
    f"Prediction time: "
    f"{prediction_time:.4f} seconds"
)


# ============================================================
# 14. INFERENCE LATENCY
# ============================================================

print("\nMeasuring inference latency")
print("----------------------------------------")

# Warm-up
_ = qat_model.predict(
    X_test[:1],
    verbose=0
)

latencies = []

for i in range(len(X_test)):

    sample = X_test[i:i + 1]

    start = time.perf_counter()

    _ = qat_model.predict(
        sample,
        verbose=0
    )

    end = time.perf_counter()

    latency_ms = (
        end - start
    ) * 1000.0

    latencies.append(
        latency_ms
    )


latencies = np.asarray(
    latencies,
    dtype=np.float64
)

mean_latency = np.mean(latencies)
median_latency = np.median(latencies)
min_latency = np.min(latencies)
max_latency = np.max(latencies)

print(
    f"Mean latency   : "
    f"{mean_latency:.4f} ms"
)

print(
    f"Median latency : "
    f"{median_latency:.4f} ms"
)

print(
    f"Min latency    : "
    f"{min_latency:.4f} ms"
)

print(
    f"Max latency    : "
    f"{max_latency:.4f} ms"
)


# ============================================================
# 15. SAVE MODEL
# ============================================================

print("\nSaving QAT model...")
print("----------------------------------------")

qat_model.save(
    MODEL_SAVE_PATH
)

print(
    f"Model saved to:\n"
    f"{MODEL_SAVE_PATH}"
)


# ============================================================
# 16. FINAL RESULTS
# ============================================================

print("\n========================================")
print("T-MLP QAT FINAL RESULTS")
print("========================================")

print(
    f"Training samples   : "
    f"{EXPECTED_TRAIN}"
)

print(
    f"Validation samples : "
    f"{EXPECTED_VAL}"
)

print(
    f"Test samples       : "
    f"{EXPECTED_TEST}"
)

print(
    f"Training windows   : "
    f"{len(X_train)}"
)

print(
    f"Validation windows : "
    f"{len(X_val)}"
)

print(
    f"Test windows       : "
    f"{len(X_test)}"
)

print(
    f"Epochs             : "
    f"{EPOCHS}"
)

print(
    f"Batch size         : "
    f"{BATCH_SIZE}"
)

print(
    f"Learning rate      : "
    f"{LEARNING_RATE}"
)

print(
    f"Test MSE           : "
    f"{test_loss:.8f}"
)

print(
    f"Test RMSE          : "
    f"{test_rmse:.8f}"
)

print(
    f"Test MAE           : "
    f"{test_mae:.8f}"
)

print(
    f"Mean latency       : "
    f"{mean_latency:.4f} ms"
)

print(
    f"Median latency     : "
    f"{median_latency:.4f} ms"
)

print(
    f"Training time      : "
    f"{training_time:.2f} sec"
)

print("========================================")


Environment
----------------------------------------
TensorFlow: 2.20.0
TFMOT     : 0.8.1

Dataset verification
----------------------------------------
Training samples   : 800
Validation samples : 100
Test samples       : 100
Dataset verification successful.

Loading training data...
Training windows: 732388

Loading validation data...
Validation windows: 80583

Loading test data...
Test windows: 85649

Building T-MLP QAT model...
----------------------------------------

Base model:
Model: "TMLP_QAT"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 noisy_audio (InputLayer)    [(None, 256)]                0         []                            
                                                                                                  
 input_reshape (Reshape)     (None, 256, 1)               0         ['noisy_audio[0][0]']     